# Frozen Idiap text and five frozen NUSD audio checkpoints on canonical DAIC-WOZ TRAIN/DEV
TEST remains unopened. No teacher fine-tuning is performed.


## 1. Mount Drive and load the pinned GitHub branch


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
if not (Path('/content/drive/MyDrive/DAIC_WOZ') / 'metadata').is_dir():
    drive.mount('/content/drive')
import base64, datetime, json, os, subprocess, sys
DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
assert (DAIC / 'metadata').is_dir(), f'Drive unavailable; reconnect Drive in Colab, then rerun: {DAIC}'
REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
BRANCH = 'feature/official141-original-teachers-80-10-10'
ROOT = Path('/content/reliability-aware-depression-kd')
CODE = Path('/content/reliability-aware-depression-kd-original188')
token = userdata.get('GITHUB_TOKEN'); assert token, 'Add GITHUB_TOKEN to Colab Secrets'
header = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy(); env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {header}')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', str(ROOT)], check=True, env=env)
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=env)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
if not CODE.exists():
    subprocess.run(['git', '-C', str(ROOT), 'worktree', 'add', '--detach', str(CODE), revision], check=True)
else:
    assert (CODE / '.git').exists(), f'Existing folder is not a Git worktree: {CODE}'


def safe_switch(target):
    """Switch this worktree, preserving dirty work before retrying a refused switch."""
    command = ['git', '-C', str(CODE), 'switch', '--detach', target]
    first = subprocess.run(command, text=True, capture_output=True)
    if first.returncode == 0:
        return
    status = subprocess.run(['git', '-C', str(CODE), 'status', '--porcelain', '--untracked-files=all'],
                            text=True, capture_output=True)
    if status.returncode != 0:
        raise RuntimeError(f'git switch failed:\n{first.stderr}\ngit status failed:\n{status.stderr}')
    if not status.stdout.strip():
        raise RuntimeError(f'git switch failed with a clean worktree. Git says:\n{first.stderr}')

    before = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '-q', '--verify', 'refs/stash'],
                            text=True, capture_output=True)
    old_stash = before.stdout.strip() if before.returncode == 0 else ''
    stash_name = 'colab-preserved-before-switch-' + target[:12]
    saved = subprocess.run(['git', '-C', str(CODE), 'stash', 'push', '--include-untracked', '-m', stash_name],
                           text=True, capture_output=True)
    if saved.returncode != 0:
        raise RuntimeError(f'Git refused the switch and could not preserve local files.\n'
                           f'Switch error: {first.stderr}\nStash error: {saved.stderr}')
    current_stash = subprocess.run(['git', '-C', str(CODE), 'rev-parse', '-q', '--verify', 'refs/stash'],
                                   text=True, capture_output=True)
    stash_sha = current_stash.stdout.strip() if current_stash.returncode == 0 else ''
    if not stash_sha or stash_sha == old_stash:
        raise RuntimeError(f'Git refused the switch, and no new stash was created.\n'
                           f'Switch error: {first.stderr}\nStash output: {saved.stdout}{saved.stderr}')

    backup_root = DAIC / 'experiments' / 'colab_worktree_backups'
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    backup = backup_root / f'{stamp}_{target[:12]}'
    backup.mkdir(parents=True, exist_ok=False)
    patch = subprocess.run(['git', '-C', str(CODE), 'stash', 'show', '--patch', '--binary',
                            '--include-untracked', stash_sha], text=True, capture_output=True)
    if patch.returncode != 0 or not patch.stdout:
        raise RuntimeError(f'Local files remain safely in Git stash {stash_sha}, but Drive backup export failed. '
                           f'No switch was attempted. Details: {patch.stderr}')
    (backup / 'local_changes.patch').write_text(patch.stdout)
    (backup / 'git_status_before_switch.txt').write_text(status.stdout)
    (backup / 'manifest.json').write_text(json.dumps({
        'target_revision': target,
        'stash_commit': stash_sha,
        'stash_name': stash_name,
        'worktree': str(CODE),
        'original_switch_error': first.stderr,
        'backup_utc': stamp,
        'restore_note': 'Changes are retained in the worktree Git stash. Inspect the patch before applying it.'
    }, indent=2) + '\n')
    clean = subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain', '--untracked-files=all'], text=True)
    if clean.strip():
        raise RuntimeError(f'Files were stashed and backed up to {backup}, but the worktree is still dirty; '
                           f'no switch was attempted. Remaining status:\n{clean}')
    retry = subprocess.run(command, text=True, capture_output=True)
    if retry.returncode != 0:
        raise RuntimeError(f'Local work is preserved in stash {stash_sha} and backup {backup}. '
                           f'The switch still failed after cleanup:\n{retry.stderr}')
    print(f'Preserved local work in Git stash {stash_sha} and Drive backup {backup}')


if subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() != revision:
    safe_switch(revision)
assert subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() == revision
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-colab.txt')], check=True)
del token, header, env
# Colab retains already imported modules from earlier revisions.
sys.path = [entry for entry in sys.path if not entry.startswith('/content/reliability-aware-depression-kd-original188')]
sys.path.insert(0, str(CODE))
for name in list(sys.modules):
    if name == 'scripts' or name.startswith('scripts.'):
        del sys.modules[name]
assert (CODE / 'scripts/experiments/internal188/coverage.py').is_file(), 'Checkout is missing coverage.py'
print('Revision:', revision)


def refresh_code():
    """Update this existing worktree; no new /content code folder."""
    secret = userdata.get('GITHUB_TOKEN')
    assert secret, 'GITHUB_TOKEN missing from Colab Secrets'
    credentials = os.environ.copy()
    credentials.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader',
                       GIT_CONFIG_VALUE_0='AUTHORIZATION: basic ' +
                       base64.b64encode(f'x-access-token:{secret}'.encode()).decode())
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=credentials)
    latest = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    current = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip()
    if latest != current:
        safe_switch(latest)
    sys.path = [entry for entry in sys.path if not entry.startswith('/content/reliability-aware-depression-kd-original188')]
    sys.path.insert(0, str(CODE))
    for name in list(sys.modules):
        if name == 'scripts' or name.startswith('scripts.'):
            del sys.modules[name]
    print('Code revision:', latest)
    del secret, credentials


## 2. Freeze the canonical 107 / 34 / 47 split


In [ ]:
import pandas as pd
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
from scripts.experiments.internal188.official_split import main as split_stage
EXP = DAIC / 'experiments' / 'idiap_ussd_canonical188_seed103'
split_stage(['--daic-root', str(DAIC), '--output', str(EXP / 'split')])
manifest = pd.read_csv(EXP / 'split' / 'manifest.csv')
assert len(manifest) == 188 and manifest.participant_id.nunique() == 188
assert 440 not in set(manifest.participant_id)
assert manifest.split.value_counts().to_dict() == {'train': 107, 'val': 34, 'student_test': 47}
print(pd.crosstab(manifest.split, manifest.label))
print('Canonical source audit:', manifest.groupby(['split', 'source_split']).size().to_dict())
print('TEST media opened: False; manifest:', EXP / 'split' / 'manifest.csv')


## 3. Frozen NUSD ECAPA-TDNN audio teacher (published DAIC-WOZ checkpoint)

This replaces the Step-Audio2 attempt. It uses the released NUSD raw ECAPA-TDNN checkpoint and the authors’ DAIC-WOZ audio preprocessing (16 kHz raw audio, speaker turns/background handling, interruption fixes, and per-file SNV). The checkpoint and published validation set have prior DAIC-WOZ exposure, so DEV results are exploratory. The preprocessing scanner is restricted to the frozen 107 TRAIN + 34 DEV IDs; TEST audio is never scanned. No training or TEST evaluation occurs in these cells.

Source repositories and the preprocessed raw-audio cache are stored under Drive (`DAIC_WOZ/tools` and `DAIC_WOZ/experiments`).


In [ ]:
refresh_code()
from scripts.experiments.internal188.nusd_ecapa import (
    NUSD_REPO, PREPROCESS_REPO, NUSD_COMMIT, PREPROCESS_COMMIT, WEIGHTS_URL,
    prepare_metadata, configure_preprocessor, configure_nusd,
    discover_checkpoint_run, export_validation_predictions,
)
import os, subprocess, sys, json, shutil, torch
TOOLS = DAIC / 'tools' / 'solo_teacher_sources'
TOOLS.mkdir(parents=True, exist_ok=True)
NUSD_SOURCE = TOOLS / 'NUSD'
PREPROCESS_SOURCE = TOOLS / 'daic_woz_process'

def checkout_pinned(repo_url, path, revision):
    if not (path / '.git').is_dir():
        subprocess.run(['git', 'clone', repo_url, str(path)], check=True)
    current = subprocess.check_output(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True).strip()
    if current != revision:
        subprocess.run(['git', '-C', str(path), 'fetch', 'origin', revision], check=True)
        subprocess.run(['git', '-C', str(path), 'checkout', '--detach', revision], check=True)
    assert subprocess.check_output(['git', '-C', str(path), 'rev-parse', 'HEAD'], text=True).strip() == revision
    print(path.name, revision)

checkout_pinned(NUSD_REPO, NUSD_SOURCE, NUSD_COMMIT)
checkout_pinned(PREPROCESS_REPO, PREPROCESS_SOURCE, PREPROCESS_COMMIT)
NUSD_EXP = EXP / 'teachers' / 'nusd_ecapa_frozen'
prepared = prepare_metadata(DAIC, EXP / 'split', NUSD_EXP)
configure_preprocessor(PREPROCESS_SOURCE, DAIC, NUSD_EXP, prepared['metadata'])
print('TRAIN/DEV metadata ready:', prepared['train_n'], prepared['dev_n'])
print('Allowed audio IDs:', len(prepared['allowed_ids']), '| TEST IDs passed to scanner: 0')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'natsort'])
print('NUSD author source and safe TRAIN/DEV-only preprocessing configured.')


## 3a. Build author-preprocessed cache, load NUSD weights, and score DEV\n\nRun this once after section 3. The public NUSD release is downloaded into the experiment cache only if no checkpoint is already present. The original runner evaluates all released runs on canonical DEV and writes participant-level probabilities, metrics, CM, and CR. It does not fine-tune weights or read TEST.\n

In [ ]:
from scripts.experiments.internal188.nusd_ecapa import patch_modern_nusd_compat
import os, subprocess, sys, json, torch
from collections import deque
assert torch.cuda.is_available(), 'Select a GPU runtime before running NUSD ECAPA inference'
feature_dir = NUSD_EXP / 'nusd_audio_cache' / 'raw_svn_exp'
feature_dir.parent.mkdir(parents=True, exist_ok=True)
cache_h5, cache_summary = feature_dir / 'complete_database.h5', feature_dir / 'summary.pickle'
if not (cache_h5.is_file() and cache_summary.is_file()):
    if feature_dir.exists():
        from datetime import datetime, timezone
        archived = feature_dir.with_name('raw_svn_exp_incomplete_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
        feature_dir.rename(archived)
        print('Preserved incomplete preprocessing output:', archived)
    env = os.environ.copy()
    env['DAIC_ALLOWED_AUDIO_MANIFEST'] = str(prepared['metadata'] / 'audio_manifest.csv')
    log_path = NUSD_EXP / 'preprocessing.log'
    tail = deque(maxlen=60)
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'run'], cwd=PREPROCESS_SOURCE,
                                   env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'NUSD preprocessing failed (exit {returncode}). Full log: {log_path}\n'
                           + ''.join(tail))
else:
    print('Using complete NUSD raw-audio cache:', feature_dir)
assert cache_h5.is_file() and cache_summary.is_file()

# Download the authors’ public pretrained NUSD checkpoint only when needed.
try:
    nusd_run, n_runs = discover_checkpoint_run(feature_dir)
except FileNotFoundError:
    import gdown
    gdown.download_folder(WEIGHTS_URL, output=str(feature_dir), quiet=False, use_cookies=False)
    nusd_run, available_runs = discover_checkpoint_run(feature_dir)
from scripts.experiments.internal188.nusd_ecapa import select_best_nusd_run
chosen, author_scores = select_best_nusd_run(nusd_run)
print('Released NUSD ECAPA author VAL macro F1 by run:', author_scores)
print('Selected one checkpoint:', chosen)
n_runs = 1
configure_nusd(NUSD_SOURCE, DAIC, NUSD_EXP, prepared['metadata'], nusd_run, n_runs)
patch_modern_nusd_compat(NUSD_SOURCE)

# The author's runner scores canonical DEV and saves one result pickle per released run.
cmd = [sys.executable, 'main_disent_fscore_grad.py', 'test', '--validate', '--cuda',
       '--prediction_metric=1', '--threshold=fscore']
log_path = NUSD_EXP / 'evaluation.log'
eval_env = os.environ.copy(); eval_env['DAIC_NUSD_FROZEN_EVAL'] = '1'
eval_env['DAIC_NUSD_RUN_INDEX'] = str(chosen['run'])
prediction_cache = nusd_run / 'results_dict_fscore' / '0.pickle'
if prediction_cache.is_file():
    print('Using saved single-run NUSD DEV predictions:', prediction_cache)
else:
    tail = deque(maxlen=80)
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', *cmd[1:]], cwd=NUSD_SOURCE, env=eval_env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'NUSD evaluation failed (exit {returncode}). Full log: {log_path}\n'
                           + ''.join(tail))
NUSD_RESULTS = NUSD_EXP / 'evaluation'
metrics = export_validation_predictions(nusd_run, NUSD_RESULTS, EXP / 'split' / 'manifest.csv', n_runs)
print('Frozen NUSD DEV metrics:', json.dumps(metrics, indent=2))
print('CM [0,1]:', metrics['confusion_matrix'])
from sklearn.metrics import classification_report
import pandas as pd
pred = pd.read_csv(NUSD_RESULTS / 'val_predictions.csv')
print(classification_report(pred.label, pred.prediction, labels=[0,1],
      target_names=['Non-depressed','Depressed'], zero_division=0))
print('Predictions:', NUSD_RESULTS / 'val_predictions.csv')


## 4. Audit TRAIN/DEV media and cached ComParE16 coverage for the text setup


In [ ]:
from scripts.experiments.internal188.coverage import main as coverage_stage
cache_candidates = [DAIC / 'experiments' / 'ussd_compare16',
                    DAIC / 'experiments' / 'full188_seed42' / 'features',
                    DAIC / 'experiments' / 'full188_80_10_10_seed103' / 'features']
available = [p for p in cache_candidates if (p / 'participant_manifest.csv').is_file()]
print('Available feature caches:', [str(p) for p in available])
assert available, 'No ComParE16 feature manifest found'
coverage_stage(['--daic-root', str(DAIC), '--split-dir', str(EXP / 'split'),
                '--output', str(EXP / 'coverage'),
                *[arg for p in available for arg in ('--feature-cache', str(p))]])
covered = pd.read_csv(EXP / 'coverage' / 'participant_manifest.csv')
assert covered.split.value_counts().to_dict() == {'train': 107, 'val': 34}
print('Coverage:', pd.crosstab(covered.split, covered.label))


## 5. Export the published frozen Idiap text checkpoint
The checkpoint is evaluated as released on canonical TRAIN-107 and DEV-34. The export uses saved training graph and DEV adjacency, excludes participant 440, and writes TRAIN logits and probabilities. No fine-tuning.


In [ ]:
refresh_code()
import json, subprocess, sys
from scripts.experiments.internal188.idiap_source import ensure_frozen_export_source
FROZEN_TEXT = EXP / 'teachers' / 'idiap_text_frozen'
text_audit = FROZEN_TEXT / 'text_target_audit.json'
text_targets = FROZEN_TEXT / 'train_text_kd_targets.csv'
if not (text_audit.is_file() and text_targets.is_file()):
    idiap_source = ensure_frozen_export_source()
    subprocess.run([sys.executable, '-m', 'scripts.teachers.idiap_text.export_targets',
                    '--daic-root', str(DAIC), '--source-root', str(idiap_source),
                    '--output', str(FROZEN_TEXT)], cwd=CODE, check=True)
pretrained_audit = json.loads(text_audit.read_text())
assert pretrained_audit['train_participants'] == 107 and pretrained_audit['dev_participants'] == 34
assert pretrained_audit['test_opened'] is False
print('Frozen Idiap TRAIN:', pretrained_audit['train_in_sample'])
print('Frozen Idiap DEV-34:', pretrained_audit['dev34'])
print('TRAIN targets:', text_targets)


The frozen Idiap checkpoint is evaluated without weight updates. Its reported DEV-34 score is distinct from the earlier `text_published_init` fine-tuning run.


## 6. Compare all five frozen NUSD checkpoints on DEV-34, then export TRAIN
Each run is evaluated separately on the same participants. The highest participant macro F1 is selected; ties use depressed F1, then AUROC. Selection on DEV makes the selected DEV score exploratory. TRAIN is in-sample. TEST remains closed.


In [ ]:
refresh_code()
import subprocess, sys, json
subprocess.run([sys.executable, '-m', 'scripts.experiments.internal188.evaluate_nusd_five',
                '--daic-root', str(DAIC), '--experiment', str(EXP), '--source', str(NUSD_SOURCE)],
               cwd=CODE, check=True)
FROZEN_AUDIO = EXP / 'teachers' / 'nusd_ecapa_frozen' / 'evaluation'
selection = json.loads((FROZEN_AUDIO / 'all_five' / 'selection.json').read_text())
print('All five DEV results:', selection['all_runs'])
print('Selected frozen audio checkpoint:', selection['selected'])
subprocess.run([sys.executable, '-m', 'scripts.experiments.internal188.export_nusd_train',
                '--daic-root', str(DAIC), '--experiment', str(EXP), '--source', str(NUSD_SOURCE)],
               cwd=CODE, check=True)
print('TRAIN:', json.loads((FROZEN_AUDIO / 'train_metrics.json').read_text()))


## 7. Plain text student setup (independent of teacher checkpoint)


In [ ]:
refresh_code()
import subprocess, sys
STUDENT = EXP / 'students' / 'canonical_v1'
BASE = [sys.executable, '-m', 'scripts.experiments.internal188.student_stages',
        '--experiment', str(EXP), '--daic-root', str(DAIC)]
subprocess.run(BASE + ['--stage', 'prepare'], cwd=CODE, check=True)
print('Student text output:', STUDENT)


In [ ]:
subprocess.run(BASE + ['--stage', 'text', '--idiap-source', str(idiap_source)], cwd=CODE, check=True)
print((STUDENT / 'plain_text' / 'metrics.json').read_text())


## 8. Teacher target alignment
Verify frozen Idiap and selected frozen NUSD TRAIN predictions cover exactly the same 107 participant IDs and labels. No teacher fine-tuning or USSD checkpoint is used.


In [ ]:
text_target = FROZEN_TEXT / 'train_text_kd_targets.csv'
audio_target = FROZEN_AUDIO / 'train_audio_targets.csv'
import pandas as pd
assert text_target.is_file() and audio_target.is_file()
t = pd.read_csv(text_target); a = pd.read_csv(audio_target)
assert len(t) == len(a) == 107 and set(t.participant_id) == set(a.participant_id)
assert t.set_index('participant_id').label.sort_index().equals(a.set_index('participant_id').label.sort_index())
assert t[['text_probability','text_logit']].notna().all().all()
assert a[['audio_probability','audio_logit']].notna().all().all()
print('Frozen teacher TRAIN-107 targets aligned; TEST CLOSED.')
